In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
import random
import sys

# =============================================================================
# 💡 튜터 코멘트: 이 데이터셋, 정말 멋지죠?
# 블랑숑/ChaBuD 데이터셋은 '변화 탐지(Change Detection)'를 위한 초고화질 위성 이미지 데이터셋이에요.
# 'ChaBuD'는 Burned Area Delineation(불탄 지역 구획)과 관련이 있습니다.
# 🛰️ 목표: 시간이 흐름에 따라 어떤 지역이, 얼마나 변화했는지 (예: 숲 -> 불탄 땅)를 AI가 알아내는 것이 목표입니다.
# 🚀 실습 목표: 데이터를 로드하고, 두 시점의 이미지를 비교하여 변화가 일어난 부분을 시각적으로 탐색해 봅시다!
# =============================================================================

# 데이터셋 정보
DATASET_NAME = "blanchon/ChaBuD"
SPLIT_NAME = "train"
# 테스트 및 실습을 위해 샘플 개수를 5개로 제한합니다. (너무 많이 로드하면 느려요!)
SAMPLE_COUNT = 5 

# 데이터셋을 로드하는 함수 (스트리밍 모드 처리 필수)
def load_dataset_safe(dataset_name: str, split_name: str, sample_count: int):
    print("=" * 80)
    print(f"💾 데이터셋 로드 시작: {dataset_name} (Split: {split_name})")
    print("=" * 80)
    
    # 1. 스트리밍 모드를 시도합니다. (가장 메모리 효율적!)
    try:
        print("👉 [시도 1/2] 스트리밍 모드(streaming=True)로 데이터셋을 로드해 보겠습니다...")
        # streaming=True는 메모리 사용량을 줄여주어 매우 큰 데이터셋에 유리합니다.
        dataset = load_dataset(dataset_name, split=split_name, streaming=True)
        print("✅ 스트리밍 로드 성공! 메모리 걱정 없이 데이터를 탐색할 준비가 되었습니다.")
        return dataset
    
    except Exception as e:
        print(f"\n⚠️ [경고] 스트리밍 로드에 실패했거나 권한 문제로 인해 예외가 발생했습니다: {e}")
        print("⬇️ [대안] 일반 Dataset 모드로, 작은 샘플만 다운로드하여 진행하겠습니다.")
        
        # 2. 실패 시, 일반 모드(streaming=False)로 소량만 다운로드합니다.
        try:
            dataset = load_dataset(dataset_name, split=split_name)
            # 전체 데이터셋을 로드할 필요 없이, 상위 K개만 테스트용으로 사용합니다.
            # 💡 필수 패턴 준수: list(dataset.take(K)) 사용
            sample_data_list = list(dataset.take(sample_count))
            print(f"✅ 일반 모드로 상위 {len(sample_data_list)}개의 샘플을 성공적으로 로드했습니다.")
            return sample_data_list
        except Exception as e_fallback:
            print(f"\n🛑 ❌ 데이터셋 로드 최종 실패. {e_fallback}")
            sys.exit("스크립트를 종료합니다. 데이터셋 경로 또는 이름 설정을 확인해 주세요.")


# 데이터셋 로드 및 테스트 데이터 준비
dataset_iterator = load_dataset_safe(DATASET_NAME, SPLIT_NAME, SAMPLE_COUNT)


def process_and_visualize(sample_data_list):
    """
    로드된 샘플 데이터 리스트를 순회하며, 변화 탐지 실습을 수행하고 시각화합니다.
    """
    print("\n" + "="*80)
    print("🧠 AI 실습 시작: 변화 탐지(Change Detection) 분석")
    print("="*80)

    # 이미지 데이터가 로드된 형태에 맞게 반복합니다.
    for i, sample in enumerate(sample_data_list):
        print(f"\n✨ [{i+1}/{len(sample_data_list)}] 샘플 분석 중...")
        
        # 1. 데이터 추출
        # 각 샘플은 'image1'(시점 1), 'image2'(시점 2), 'mask'(정답)를 포함합니다.
        img1_pil = sample['image1']
        img2_pil = sample['image2']
        mask_pil = sample['mask']

        # 2. PIL 이미지 객체를 NumPy 배열로 변환 (AI 연산을 위해 필수!)
        # 💡 NumPy 배열은 [높이, 너비, 채널] 형태를 가집니다.
        try:
            img1 = np.array(img1_pil)
            img2 = np.array(img2_pil)
            mask = np.array(mask_pil)
        except Exception as e:
            print(f"   [ERROR] 이미지 배열 변환 오류: {e}. 이 샘플은 건너뜁니다.")
            continue

        # 3. 핵심 실습: 변화 계산 (Feature Engineering)
        # 변화는 '두 시점의 이미지 차이'로 근사할 수 있습니다. (L1 Norm 근사)
        # 모든 채널(R, G, B)을 비교하여 절대값 차이를 구합니다.
        # (H, W, 3) 배열의 차이를 계산합니다.
        change_map = np.abs(img1 - img2)
        
        print(f"   🖼️ 이미지 크기: {img1.shape[0]}x{img1.shape[1]} (H x W)")
        print(f"   📊 계산된 변화 맵 크기: {change_map.shape[0]}x{change_map.shape[1]}x{change_map.shape[2]}")
        print("   🧪 변화 계산 완료: |Image1 - Image2|를 통해 변화의 크기를 측정했습니다.")
        
        # 4. 결과 시각화 (matplotlib 사용)
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # --- 1. 시점 1 (Image 1) ---
        axes[0].imshow(img1 / 255.0) # 데이터 정규화하여 색상 범위 조정 (0~1)
        axes[0].set_title("Time 1 Image (Image1)")
        axes[0].axis('off')
        
        # --- 2. 시점 2 (Image 2) ---
        axes[1].imshow(img2 / 255.0)
        axes[1].set_title("Time 2 Image (Image2)")
        axes[1].axis('off')
        
        # --- 3. 변화 분석 및 Ground Truth 비교 ---
        # 변화 맵의 평균 강도를 이용하거나, 가장 극단적으로 차이가 난 채널만 시각화합니다.
        # 여기서는 'change_map'의 평균을 사용해 변화가 얼마나 심했는지 간략히 표현합니다.
        # (진정한 변화 탐지는 여기에 고급 알고리즘이 들어갑니다!)
        axes[2].imshow(np.mean(change_map, axis=2) / 255.0, cmap='viridis') 
        axes[2].set_title("Calculated Change Map (Intensity)")
        axes[2].axis('off')

        plt.suptitle(f"Sample {i+1}: Change Detection Visualization", fontsize=16)
        plt.show()


# ======= 실습 실행 =======
process_and_visualize(dataset_iterator)

print("\n\n🎉 축하합니다! 데이터셋 구조 파악과 기초 변화 탐지 시각화 실습을 모두 완료했습니다!")
print("이 코드를 기반으로, 'Mask'와 'Change Map'의 상관관계를 분석하는 다음 단계로 나아갈 수 있습니다.")